## **Implementing Multiple Linear Regression from Scratch**

### **Topic Roadmap**
- **2. Train-Test Split**
- **3. Baseline Model (Scikit-Learn)**
- **4. Mathematical Formulation & Class Definition**
- **5. Custom Model Evaluation**
- **6. Key Revision Notes**

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
# Load diabetes dataset (X: features, y: target)
X, y = load_diabetes(return_X_y=True)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

Feature matrix shape: (442, 10)
Target vector shape: (442,)


## **2. Train-Test Split**

Split the data into 80% training and 20% testing sets to ensure fair evaluation of the models[cite: 18].

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print(f"Training instances: {X_train.shape[0]}")
print(f"Testing instances: {X_test.shape[0]}")

Training instances: 353
Testing instances: 89


## **3. Baseline Model (Scikit-Learn)**

Train a standard `LinearRegression` model using `scikit-learn` to serve as a baseline. We will extract its $R^2$ score, coefficients, and intercept to verify our custom implementation later[cite: 18].

In [4]:
# Initialize and train the scikit-learn model
reg_sklearn = LinearRegression()
reg_sklearn.fit(X_train, y_train)

# Predict and evaluate
y_pred_sklearn = reg_sklearn.predict(X_test)
sklearn_r2 = r2_score(y_test, y_pred_sklearn)

print(f"Scikit-Learn R2 Score: {sklearn_r2:.6f}")
print(f"Scikit-Learn Intercept: {reg_sklearn.intercept_:.6f}")
print(f"Scikit-Learn Coefficients: \n{reg_sklearn.coef_}")

Scikit-Learn R2 Score: 0.439934
Scikit-Learn Intercept: 151.883310
Scikit-Learn Coefficients: 
[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]


## **4. Mathematical Formulation & Class Definition**

In Multiple Linear Regression, we compute the coefficients analytically using the **Normal Equation**[cite: 18]:

$$\beta = (X^TX)^{-1} X^Ty$$

Where:
- $X$ is the input feature matrix (with a column of 1s added to account for the intercept).
- $y$ is the target vector.
- $\beta$ contains the intercept (at index 0) and the feature coefficients (indices 1 to n).

We implement this matrix multiplication using `numpy.linalg.inv` and `numpy.dot`[cite: 18].

In [5]:
class CustomMultipleLR:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # Add a column of 1s at index 0 for the intercept term
        X_train_augmented = np.insert(X_train, 0, 1, axis=1)

        # Apply the Normal Equation: Beta = (X^T * X)^-1 * X^T * y
        betas = np.linalg.inv(np.dot(X_train_augmented.T, X_train_augmented)).dot(X_train_augmented.T).dot(y_train)
        
        # Extract intercept (first element) and coefficients (remaining elements)
        self.intercept_ = betas[0]
        self.coef_ = betas[1:]

    def predict(self, X_test):
        # Apply the linear equation: y_pred = X * coef + intercept
        y_pred = np.dot(X_test, self.coef_) + self.intercept_
        return y_pred

## **5. Custom Model Evaluation**

Instantiate the custom class, fit it on the training data, and compare its predictions and parameters directly against the Scikit-Learn baseline[cite: 18].

In [6]:
# Initialize and train the custom model
lr_custom = CustomMultipleLR()
lr_custom.fit(X_train, y_train)

# Predict and evaluate
y_pred_custom = lr_custom.predict(X_test)
custom_r2 = r2_score(y_test, y_pred_custom)

print(f"Custom Model R2 Score: {custom_r2:.6f}")
print(f"Custom Model Intercept: {lr_custom.intercept_:.6f}")
print(f"Custom Model Coefficients: \n{lr_custom.coef_}")

Custom Model R2 Score: 0.439934
Custom Model Intercept: 151.883310
Custom Model Coefficients: 
[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]


## **Key Revision Notes**

- **Normal Equation:** The analytical method for finding the exact coefficients in Multiple Linear Regression without relying on gradient descent optimization[cite: 18]. It computes $(X^TX)^{-1} X^Ty$.
- **Intercept Handling:** To calculate the y-intercept simultaneously with the slopes, we artificially prepend a column of $1$s to the $X$ matrix using `np.insert(X_train, 0, 1, axis=1)`[cite: 18]. The first value in the resulting weight vector $\beta$ becomes the intercept.
- **Matrix Computations in Numpy:**
  - Matrix Transpose: `X.T`[cite: 18]
  - Dot Product: `np.dot(A, B)`[cite: 18]
  - Matrix Inverse: `np.linalg.inv(A)`[cite: 18]
- **Performance Constraint:** Computing the inverse of an $n \times n$ matrix is computationally expensive $O(n^3)$. While the normal equation works flawlessly for small to medium feature sets (like the diabetes dataset), it becomes critically slow if the number of features is extremely large (e.g., > 10,000).